# BBBC021: balanced five-class patch dataset

This notebook creates 200,000 balanced 224 by 224 three-channel patches for classification and Grad-CAM. The selected classes are DMSO, cytochalasin B, nocodazole, taxol and AZ-A.

## 1. Experimental unit and split strategy

The biological hierarchy is `compound → concentration → plate → well → field of view → patch`. Splits are assigned by plate and well before patch extraction. Patch count must therefore not be interpreted as the number of independent biological observations.

## 2. Setup and configuration

Install the repository with `python -m pip install -e ".[bbbc021]"`. In Colab, set `work_dir` to a writable path such as `/content/bbbc021_5class`.

In [ ]:
from dataclasses import asdict
from pathlib import Path

import matplotlib.pyplot as plt

from bbbc021_patch_extraction.pipeline import (
    PatchExtractionConfig,
    allocate_patches,
    assign_splits,
    create_paths,
    download_source_images,
    generate_dataset,
    load_selected_metadata,
    pilot_storage_estimate,
    source_summary,
    validate_manifest,
)

config = PatchExtractionConfig(work_dir=Path("bbbc021_5class"))
paths = create_paths(config)
asdict(config)

## 3. Select source metadata

BBBC021 is a fixed 24-hour endpoint assay, not a time series. One metadata row represents one field of view with DAPI, actin and tubulin image files.

In [ ]:
metadata = load_selected_metadata(config, paths)
metadata = assign_splits(metadata, config)
source_summary(metadata)

## 4. Download required plate archives

The source images are distributed by plate. Only TIFF files referenced by the selected compound metadata are extracted.

In [ ]:
archive_manifest = download_source_images(metadata, config, paths)
archive_manifest
print(f"Required source download: {archive_manifest['GiB'].sum():.2f} GiB")

## 5. Pilot storage estimate

Channels are ordered DAPI, actin and tubulin. Each channel is percentile-scaled to uint8 before lossless PNG encoding. The pilot is more informative than a raw-array estimate because compressibility depends on image content.

In [ ]:
storage_estimate = pilot_storage_estimate(metadata, config, paths)
storage_estimate

## 6. Balanced patch allocation

Every class receives the same number of patches. Within each class, counts follow the train, validation and test fractions.

In [ ]:
allocation = allocate_patches(config)
allocation.pivot(index="class_name", columns="split", values="patches")

## 7. Generate WebDataset shards

Exact duplicate coordinates within a field of view are rejected. Overlapping patches can still be correlated, especially for classes with few source fields.

In [ ]:
patch_manifest = generate_dataset(metadata, allocation, config, paths)
print(f"Wrote {len(patch_manifest):,} patches to {paths.output}")

## 8. Validate biological separation and output balance

Validation checks the requested total, class balance and absence of plate-well leakage between splits.

In [ ]:
summary = validate_manifest(patch_manifest, config)
summary

axis = patch_manifest.groupby(["class_name", "split"]).size().unstack(fill_value=0).plot(
    kind="bar", stacked=True, figsize=(10, 5)
)
axis.set_ylabel("patches")
axis.set_title("Patch allocation")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()

## 9. Output and interpretation

The output contains TAR shards, `manifest.csv` and `config.json`. TAR shards avoid the overhead of copying 200,000 individual files. For downstream evaluation, report performance both per patch and after aggregation by source field or well.